In [ ]:
# Accelerator: GPU T4 x2
!nvidia-smi

# Neural Debris Removal

Goal: de-poison the provided RetinaNet model and submit predictions that are closer to the hidden clean model.

Local validation is only a proxy. We use it to avoid wasting Kaggle submissions.

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

### Imports

In [ ]:
import copy, json, logging, os, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, DatasetMapper, MetadataCatalog
from detectron2.data import build_detection_train_loader, detection_utils as utils
from detectron2.engine import DefaultPredictor, DefaultTrainer

logging.getLogger("detectron2").setLevel(logging.WARNING)
logging.getLogger("fvcore").setLevel(logging.WARNING)

### Constants

In [ ]:
ROOT = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"
POISONED_WEIGHTS = f"{ROOT}/poisoned_model/poisoned_model.pth"
UNLEARN_DIR = f"{ROOT}/unlearn_set"
TEST_DIR = f"{ROOT}/test_set/test_set"

BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1

BATCH_SIZE = 4
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024

UNLEARN_DATASET = "unlearn"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

unlearn_files = sorted(Path(UNLEARN_DIR).glob("*.png"))
test_files = sorted(Path(TEST_DIR).glob("*.png"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Global helpers

In [ ]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def build_base_cfg(weights, output_dir=None, score_thresh=CONF_THRESH):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    if output_dir is not None:
        cfg.OUTPUT_DIR = output_dir
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg

def build_predictor(weights, score_thresh=CONF_THRESH):
    return DefaultPredictor(build_base_cfg(weights, score_thresh=score_thresh))

### Dataset and trainer

In [ ]:
def register_unlearn_dataset():
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)

    with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
        coco = json.load(f)

    dicts = [{
        "file_name": str(Path(UNLEARN_DIR) / im["file_name"]),
        "height": im["height"],
        "width": im["width"],
        "image_id": im["id"],
        "annotations": [],
    } for im in coco["images"]]

    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print("Registered unlearn images:", len(dicts))

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

class UnlearnTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

    @classmethod
    def build_model(cls, cfg):
        logging.getLogger("detectron2").setLevel(logging.ERROR)
        return super().build_model(cfg)

register_unlearn_dataset()

### Validation

In [ ]:
def sample_files(files, n=None, seed=SEED):
    files = list(files)
    if n is None or n >= len(files):
        return files
    rng = random.Random(seed)
    return rng.sample(files, n)

def evaluate_predictor(predictor, files):
    total_boxes = 0
    total_conf = 0.0
    empty_count = 0
    all_scores = []
    for img_path in tqdm(files, desc="Validation"):
        scores = predictor(load_image(img_path))["instances"].to("cpu").scores.numpy()
        total_boxes += len(scores)
        total_conf += scores.sum()
        empty_count += int(len(scores) == 0)
        all_scores.extend(scores.tolist())

    n = len(files)
    return {
        "images": n,
        "total_boxes": total_boxes,
        "mean_boxes_per_image": total_boxes / n,
        "mean_conf_sum_per_image": float(total_conf / n),
        "empty_rate": empty_count / n,
        "mean_score_per_box": float(np.mean(all_scores)) if all_scores else 0.0,
        "median_score": float(np.median(all_scores)) if all_scores else 0.0,
    }

def evaluate_model(weights, n_test=300, seed=SEED):
    predictor = build_predictor(weights)
    poison_stats = evaluate_predictor(
        predictor,
        unlearn_files
    )
    test_stats = evaluate_predictor(
        predictor,
        sample_files(test_files, n_test, seed)
    )
    return {
        "poison_mean_boxes": poison_stats["mean_boxes_per_image"],
        "poison_mean_conf": poison_stats["mean_conf_sum_per_image"],
        "poison_empty_rate": poison_stats["empty_rate"],
        "test_mean_boxes": test_stats["mean_boxes_per_image"],
        "test_mean_conf": test_stats["mean_conf_sum_per_image"],
        "test_empty_rate": test_stats["empty_rate"],
    }

# 1. Quick scoring improvements

## 1.1. LR sweep

In [ ]:
def build_unlearn_cfg(name, lr, iters):
    cfg = build_base_cfg(
        POISONED_WEIGHTS,
        output_dir=f"/kaggle/working/{name}"
    )
    cfg.DATASETS.TRAIN = (UNLEARN_DATASET,)
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = lr
    cfg.SOLVER.MAX_ITER = iters
    cfg.SOLVER.STEPS = []
    return cfg

def train_unlearn_model(name, lr, iters):
    cfg = build_unlearn_cfg(name, lr, iters)
    trainer = UnlearnTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return str(Path(cfg.OUTPUT_DIR) / "model_final.pth")

def run_lr_experiments(experiments):
    rows = []
    for exp in experiments:
        weights = train_unlearn_model(**exp)
        row = {
            "name": exp["name"],
            "lr": exp["lr"],
            "iters": exp["iters"],
            **evaluate_model(weights)
        }
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
experiments = [
    # Original baseline / sanity-check experiments
    {"name": "lr1e-4_iter20", "lr": 1e-4, "iters": 20},
    # {"name": "lr1e-4_iter50", "lr": 1e-4, "iters": 50},
    {"name": "lr5e-5_iter50", "lr": 5e-5, "iters": 50},

    # Focused sweep around the current promising region
    # {"name": "lr1.5e-4_iter20", "lr": 1.5e-4, "iters": 20},
    # {"name": "lr1.75e-4_iter20", "lr": 1.75e-4, "iters": 20},
    # {"name": "lr2e-4_iter20", "lr": 2e-4, "iters": 20},
    {"name": "lr2.25e-4_iter20", "lr": 2.25e-4, "iters": 20},
    {"name": "lr2.5e-4_iter20", "lr": 2.5e-4, "iters": 20},

    # Aggressive reference from the first experiments
    # {"name": "lr2e-4_iter50", "lr": 2e-4, "iters": 50},
]
results_df = run_lr_experiments(experiments)
results_df

Increasing the learning rate improved poison suppression. The best trade-off is achieved around `lr=2e-4` to `2.5e-4` with `20` iterations.

Training for `50` iterations caused significant over-unlearning and degraded normal detections.

## 1.2. Confidence threshold sweep

In [ ]:
def evaluate_model_th(weights, score_thresh, n_test=300, seed=SEED):
    predictor = build_predictor(weights, score_thresh=score_thresh)
    poison = evaluate_predictor(predictor, unlearn_files)
    test = evaluate_predictor(predictor, sample_files(test_files, n_test, seed))
    return {
        "poison_mean_boxes": poison["mean_boxes_per_image"],
        "poison_mean_conf": poison["mean_conf_sum_per_image"],
        "poison_empty_rate": poison["empty_rate"],
        "test_mean_boxes": test["mean_boxes_per_image"],
        "test_mean_conf": test["mean_conf_sum_per_image"],
        "test_empty_rate": test["empty_rate"],
    }

def run_threshold_experiments(candidate_models, thresholds):
    rows = []
    for model_name, weights in candidate_models.items():
        for threshold in thresholds:
            rows.append({
                "model": model_name,
                "threshold": threshold,
                **evaluate_model_th(weights, threshold)
            })
    return pd.DataFrame(rows)

In [ ]:
candidate_models = {
    # "lr2e-4_iter20": "/kaggle/working/lr2e-4_iter20/model_final.pth",
    # "lr2.25e-4_iter20": "/kaggle/working/lr2.25e-4_iter20/model_final.pth",
    "lr2.5e-4_iter20": "/kaggle/working/lr2.5e-4_iter20/model_final.pth",
}
thresholds = [0.15, 0.18, 0.20, 0.22, 0.25, 0.30]
threshold_results_df = run_threshold_experiments(candidate_models, thresholds)
threshold_results_df

In [ ]:
threshold_results_df.sort_values(
    by=["poison_mean_conf", "test_empty_rate"],
    ascending=[True, True]
)

Increasing the confidence threshold reduced poison detections further, but high thresholds caused too many empty test predictions.

The best local trade-off is `lr2.5e-4_iter20` with threshold `0.22`: strong poison suppression while keeping test degradation moderate.

## 1.3. Weight interpolation between poisoned and unlearned weights

In [ ]:
def get_model_state(path):
    ckpt = torch.load(path, map_location="cpu")
    return ckpt["model"] if "model" in ckpt else ckpt

def save_interpolated_model(poisoned_path, unlearned_path, alpha, out_path):
    poisoned_ckpt = torch.load(poisoned_path, map_location="cpu")
    unlearned_ckpt = torch.load(unlearned_path, map_location="cpu")
    poisoned_state = poisoned_ckpt["model"] if "model" in poisoned_ckpt else poisoned_ckpt
    unlearned_state = unlearned_ckpt["model"] if "model" in unlearned_ckpt else unlearned_ckpt

    new_state = {}
    for k in poisoned_state:
        if k in unlearned_state and torch.is_tensor(poisoned_state[k]) and poisoned_state[k].shape == unlearned_state[k].shape:
            new_state[k] = poisoned_state[k] + alpha * (unlearned_state[k] - poisoned_state[k])
        else:
            new_state[k] = poisoned_state[k]

    out_ckpt = dict(poisoned_ckpt)
    out_ckpt["model"] = new_state
    torch.save(out_ckpt, out_path)
    return out_path

In [ ]:
base_unlearned_model = "/kaggle/working/lr2.5e-4_iter20/model_final.pth"
alphas = [0.4, 0.6, 0.8, 1.0, 1.2]

rows = []
for alpha in alphas:
    name = f"interp_alpha{alpha}"
    out_path = f"/kaggle/working/{name}.pth"
    save_interpolated_model(
        poisoned_path=POISONED_WEIGHTS,
        unlearned_path=base_unlearned_model,
        alpha=alpha,
        out_path=out_path
    )
    rows.append({
        "name": name,
        "alpha": alpha,
        **evaluate_model_th(out_path, score_thresh=0.22)
    })

interpolation_results_df = pd.DataFrame(rows)
interpolation_results_df

In [ ]:
interpolation_results_df.sort_values(
    by=["poison_mean_conf", "test_empty_rate"],
    ascending=[True, True]
)

Weight interpolation did not improve over the best threshold result.

Lower alpha values were too close to the poisoned model, while `alpha=1.2` caused too much test degradation.

Current best local candidate remains `lr2.5e-4_iter20` with threshold `0.22`.

# 2. Model-training improvements

## 2.1. Freeze backbone and train only detection heads

In [ ]:
class FreezeBackboneTrainer(UnlearnTrainer):
    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)

        for name, param in model.named_parameters():
            if name.startswith("backbone."):
                param.requires_grad = False

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

        print(f"Trainable params: {trainable:,}")
        print(f"Frozen params: {frozen:,}")
        return model

In [ ]:
def train_freeze_backbone_model(name, lr, iters):
    cfg = build_unlearn_cfg(name, lr, iters)
    trainer = FreezeBackboneTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return str(Path(cfg.OUTPUT_DIR) / "model_final.pth")

def run_freeze_backbone_experiments(experiments):
    rows = []
    for exp in experiments:
        weights = train_freeze_backbone_model(**exp)
        rows.append({
            "name": exp["name"],
            "lr": exp["lr"],
            "iters": exp["iters"],
            **evaluate_model(weights)
        })
    return pd.DataFrame(rows)

In [ ]:
freeze_backbone_experiments = [
    {"name": "freeze_backbone_lr2.5e-4_iter20", "lr": 2.5e-4, "iters": 20},
]
freeze_backbone_results_df = run_freeze_backbone_experiments(freeze_backbone_experiments)
freeze_backbone_results_df

Freezing the backbone preserved normal test predictions better, but poison suppression was much weaker than the full fine-tuning model.

The backbone-only freezing strategy is too conservative for now for current setup.

## 2.2. Train classifier head only

In [ ]:
class ClassifierOnlyTrainer(UnlearnTrainer):
    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)

        for name, param in model.named_parameters():
            param.requires_grad = False
            if "cls_score" in name or "cls_subnet" in name:
                param.requires_grad = True

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

        print(f"Trainable params: {trainable:,}")
        print(f"Frozen params: {frozen:,}")
        return model

In [ ]:
def train_classifier_only_model(name, lr, iters):
    cfg = build_unlearn_cfg(name, lr, iters)
    trainer = ClassifierOnlyTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return str(Path(cfg.OUTPUT_DIR) / "model_final.pth")

def run_classifier_only_experiments(experiments):
    rows = []
    for exp in experiments:
        weights = train_classifier_only_model(**exp)
        rows.append({"name": exp["name"], "lr": exp["lr"], "iters": exp["iters"], **evaluate_model(weights)})
    return pd.DataFrame(rows)

In [ ]:
classifier_only_experiments = [
    {"name": "cls_only_lr2.5e-4_iter20", "lr": 2.5e-4, "iters": 20},
]
classifier_only_results_df = run_classifier_only_experiments(classifier_only_experiments)
classifier_only_results_df

In [ ]:
classifier_only_experiments = [
    {"name": "cls_only_lr5e-4_iter20", "lr": 5e-4, "iters": 20},
]
classifier_only_results_df_2 = run_classifier_only_experiments(classifier_only_experiments)
classifier_only_results_df_2

In [ ]:
classifier_only_experiments = [
    {"name": "cls_only_lr5e-4_iter10", "lr": 5e-4, "iters": 10},
]

classifier_only_iter10_df = run_classifier_only_experiments(
    classifier_only_experiments
)
classifier_only_iter10_df

### A threshold sweep on classifier-only

In [ ]:
candidate_models = {
    "cls_only_lr5e-4_iter20": "/kaggle/working/cls_only_lr5e-4_iter20/model_final.pth",
}
thresholds = [0.10, 0.12, 0.15, 0.18, 0.20, 0.22]
cls_threshold_results_df = run_threshold_experiments(candidate_models, thresholds)
cls_threshold_results_df

Classifier-only training with a higher learning rate became competitive after threshold tuning.

The best local trade-off is `cls_only_lr5e-4_iter20` with threshold `0.15`: poison suppression is close to the best full fine-tuning model, while test degradation is slightly lower.

## 2.3. Add augmentation during unlearning

### 2.3.1. EDA before augmentation

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
    coco = json.load(f)

images_df = pd.DataFrame(coco["images"])
ann_df = pd.DataFrame(coco["annotations"])
print(images_df.shape)
print(ann_df.shape)
ann_df.head()

In [ ]:
ann_df["w"] = ann_df["bbox"].apply(lambda x: x[2])
ann_df["h"] = ann_df["bbox"].apply(lambda x: x[3])
ann_df["area"] = ann_df["w"] * ann_df["h"]
ann_df["aspect_ratio"] = ann_df["w"] / ann_df["h"]
ann_df[["w", "h", "area", "aspect_ratio"]].describe()

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
axes = axes.flatten()

for ax, img_info in zip(axes, coco["images"]):
    img_path = Path(UNLEARN_DIR) / img_info["file_name"]
    img = load_image(img_path)

    anns = [a for a in coco["annotations"] if a["image_id"] == img_info["id"]]

    ax.imshow(img[:, :, 0], cmap="gray")

    for ann in anns:
        x, y, w, h = ann["bbox"]

        ax.add_patch(
            plt.Rectangle(
                (x, y), w, h,
                fill=False,
                edgecolor="lime",
                linewidth=2
            )
        )

    ax.set_title(img_info["file_name"], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ann_df["w"].hist(ax=ax[0], bins=20)
ax[0].set_title("Width")

ann_df["h"].hist(ax=ax[1], bins=20)
ax[1].set_title("Height")

ann_df["aspect_ratio"].hist(ax=ax[2], bins=20)
ax[2].set_title("Aspect Ratio")

plt.tight_layout()
plt.show()

In [ ]:
ann_df["cx"] = ann_df["bbox"].apply(lambda x: x[0] + x[2] / 2)
ann_df["cy"] = ann_df["bbox"].apply(lambda x: x[1] + x[3] / 2)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ann_df["cx"], ann_df["cy"])

ax.set_xlim(0, 1024)
ax.set_ylim(1024, 0)

ax.set_title("Box centers")
plt.show()

The poison objects are small (typically 20-60 px) and are distributed across the entire image rather than concentrated in a specific region.

Most objects are slightly wider than tall, suggesting a preference toward horizontal streaks, although vertical examples also exist.

Augmentation plan to try:
1. Brightness + Contrast
2. Brightness + Contrast + Horizontal Flip

### 2.3.2. Add augmentation

In [ ]:
from detectron2.data import transforms as T

In [ ]:
class AugMapper(DatasetMapper):
    def __init__(self, cfg, augmentations):
        super().__init__(cfg, is_train=True)
        self.augs = T.AugmentationList(augmentations)

    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        aug_input = T.AugInput(image)
        self.augs(aug_input)
        dataset_dict["image"] = torch.as_tensor(
            aug_input.image.transpose(2, 0, 1).copy()
        )
        dataset_dict["instances"] = utils.annotations_to_instances([], aug_input.image.shape[:2])
        return dataset_dict

class AugTrainer(UnlearnTrainer):
    AUGS = []
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = AugMapper(cfg, cls.AUGS)
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

In [ ]:
def train_aug_model(name, lr, iters, augmentations):
    AugTrainer.AUGS = augmentations
    cfg = build_unlearn_cfg(name, lr, iters)
    trainer = AugTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return str(Path(cfg.OUTPUT_DIR) / "model_final.pth")

def run_aug_experiments(experiments):
    rows = []
    for exp in experiments:
        weights = train_aug_model(**exp)
        rows.append({
            "name": exp["name"],
            "lr": exp["lr"],
            "iters": exp["iters"],
            **evaluate_model(weights)
        })
    return pd.DataFrame(rows)

In [ ]:
aug_experiments = [
    {"name": "aug_bc_lr2.5e-4_iter20", "lr": 2.5e-4, "iters": 20,
     "augmentations": [T.RandomBrightness(0.9, 1.1), T.RandomContrast(0.9, 1.1)]},

    {"name": "aug_bc_hflip_lr2.5e-4_iter20", "lr": 2.5e-4, "iters": 20,
     "augmentations": [T.RandomBrightness(0.9, 1.1), T.RandomContrast(0.9, 1.1),
                       T.RandomFlip(prob=0.5, horizontal=True)]},
]
aug_results_df = run_aug_experiments(aug_experiments)
aug_results_df

In [ ]:
aug_experiments_30 = [
    {"name": "aug_bc_hflip_lr2.5e-4_iter30", "lr": 2.5e-4, "iters": 30,
     "augmentations": [T.RandomBrightness(0.9, 1.1), T.RandomContrast(0.9, 1.1),
                       T.RandomFlip(prob=0.5, horizontal=True)]},
]
aug_results_30_df = run_aug_experiments(aug_experiments_30)
aug_results_30_df

Extending augmented training to 30 iterations caused clear over-unlearning.

Although poison detections were almost fully removed, normal test predictions collapsed too much.

The 20-iteration augmentation models remain safer, but they do not outperform the best classifier-only candidate.

## 2.4. Targeted crop unlearning

In [ ]:
CROP_DATASET = "unlearn_crops"

def make_crop_dataset(crop_size=256):
    with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
        coco = json.load(f)

    img_by_id = {im["id"]: im for im in coco["images"]}
    records = []
    for ann in coco["annotations"]:
        im = img_by_id[ann["image_id"]]
        x, y, w, h = ann["bbox"]
        cx, cy = x + w / 2, y + h / 2

        x0 = int(np.clip(cx - crop_size / 2, 0, IMG_W - crop_size))
        y0 = int(np.clip(cy - crop_size / 2, 0, IMG_H - crop_size))

        records.append({
            "file_name": str(Path(UNLEARN_DIR) / im["file_name"]),
            "image_id": ann["id"],
            "height": crop_size,
            "width": crop_size,
            "crop_xy": (x0, y0),
            "annotations": [],
        })
    return records

def register_crop_dataset(crop_size=256):
    if CROP_DATASET in DatasetCatalog:
        DatasetCatalog.remove(CROP_DATASET)
        
    records = make_crop_dataset(crop_size)
    DatasetCatalog.register(CROP_DATASET, lambda: records)
    MetadataCatalog.get(CROP_DATASET).set(thing_classes=["object"])
    print("Registered crops:", len(records), "crop_size:", crop_size)

In [ ]:
class CropUnlearnMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        x0, y0 = dataset_dict["crop_xy"]
        image = image[y0:y0 + dataset_dict["height"], x0:x0 + dataset_dict["width"]]

        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

class CropUnlearnTrainer(UnlearnTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = CropUnlearnMapper(cfg, is_train=True, augmentations=[])
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

In [ ]:
def build_crop_unlearn_cfg(name, lr, iters):
    cfg = build_base_cfg(POISONED_WEIGHTS, output_dir=f"/kaggle/working/{name}")
    cfg.DATASETS.TRAIN = (CROP_DATASET,)
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = lr
    cfg.SOLVER.MAX_ITER = iters
    cfg.SOLVER.STEPS = []
    return cfg

def train_crop_unlearn_model(name, lr, iters, crop_size=256):
    register_crop_dataset(crop_size)
    cfg = build_crop_unlearn_cfg(name, lr, iters)
    trainer = CropUnlearnTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return str(Path(cfg.OUTPUT_DIR) / "model_final.pth")

def run_crop_experiments(experiments):
    rows = []
    for exp in experiments:
        weights = train_crop_unlearn_model(**exp)
        rows.append({"name": exp["name"], "lr": exp["lr"], "iters": exp["iters"], "crop_size": exp["crop_size"], **evaluate_model(weights)})
    return pd.DataFrame(rows)

In [ ]:
crop_experiments = [
    {"name": "crop256_lr2.5e-4_iter20", "lr": 2.5e-4, "iters": 20, "crop_size": 256},
    {"name": "crop256_lr5e-4_iter20", "lr": 5e-4, "iters": 20, "crop_size": 256},
]
crop_results_df = run_crop_experiments(crop_experiments)
crop_results_df

In [ ]:
crop_experiments = [
    {"name": "crop256_lr1e-4_iter20", "lr": 1e-4, "iters": 20, "crop_size": 256},
    {"name": "crop384_lr1e-4_iter20", "lr": 1e-4, "iters": 20, "crop_size": 384},
]
crop_results_df2 = run_crop_experiments(crop_experiments)
crop_results_df2

In [ ]:
crop_experiments = [
    {"name": "crop256_lr1.5e-4_iter20", "lr": 1.5e-4, "iters": 20, "crop_size": 256},
    {"name": "crop256_lr2e-4_iter20", "lr": 2e-4, "iters": 20, "crop_size": 256},
]
crop_results_df3 = run_crop_experiments(crop_experiments)
crop_results_df3

### threshold sweep on both crop models

In [ ]:
candidate_models = {
    # "crop256_lr1.5e-4_iter20": "/kaggle/working/crop256_lr1.5e-4_iter20/model_final.pth",
    "crop256_lr2e-4_iter20": "/kaggle/working/crop256_lr2e-4_iter20/model_final.pth",
}
thresholds = [0.12, 0.15, 0.18, 0.20, 0.22, 0.25]
crop_threshold_results_df = run_threshold_experiments(candidate_models, thresholds)
crop_threshold_results_df

Crop unlearning preserved normal detections well, but did not clearly outperform the best classifier-only model.

## 2.5. Global pruning

In [ ]:
def save_globally_pruned_model(weights, amount, out_path):
    ckpt = torch.load(weights, map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt

    values = [
        v.abs().flatten()
        for v in state.values()
        if torch.is_tensor(v) and v.dtype.is_floating_point and v.ndim > 1
    ]
    all_values = torch.cat(values)

    k = max(1, int(amount * all_values.numel()))
    threshold = torch.kthvalue(all_values, k).values

    new_state = {}
    for k, v in state.items():
        if torch.is_tensor(v) and v.dtype.is_floating_point and v.ndim > 1:
            new_state[k] = v * (v.abs() > threshold)
        else:
            new_state[k] = v

    out_ckpt = dict(ckpt)
    out_ckpt["model"] = new_state
    torch.save(out_ckpt, out_path)
    return out_path

In [ ]:
prune_base_model = "/kaggle/working/lr2.5e-4_iter20/model_final.pth"

prune_experiments = [
    # {"name": "prune1pct_lr2.5e-4_iter20", "amount": 0.01},
    # {"name": "prune3pct_lr2.5e-4_iter20", "amount": 0.03},
    {"name": "prune5pct_lr2.5e-4_iter20", "amount": 0.05},
]

rows = []
for exp in prune_experiments:
    out_path = f"/kaggle/working/{exp['name']}.pth"
    weights = save_globally_pruned_model(prune_base_model, exp["amount"], out_path)
    rows.append({"name": exp["name"], "amount": exp["amount"], **evaluate_model_th(weights, 0.22)})

prune_results_df = pd.DataFrame(rows)
prune_results_df

Small global pruning had almost no effect on poison suppression or normal test predictions for the current solution.

## 2.6. Retention-aware unlearning

In [ ]:
from detectron2.structures import BoxMode

RETENTION_DATASET = "retention_unlearn"

def get_teacher_annotations(predictor, img_path, score_thresh=0.25, max_boxes=20):
    out = predictor(load_image(img_path))["instances"].to("cpu")
    boxes = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()
    anns = []
    for (x1, y1, x2, y2), score in zip(boxes, scores):
        if score < score_thresh:
            continue

        x1, y1 = float(np.clip(x1, 0, IMG_W)), float(np.clip(y1, 0, IMG_H))
        x2, y2 = float(np.clip(x2, 0, IMG_W)), float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w > 0 and h > 0:
            anns.append({
                "bbox": [x1, y1, w, h],
                "bbox_mode": BoxMode.XYWH_ABS,
                "category_id": 0,
            })
        if len(anns) >= max_boxes:
            break
    return anns

In [ ]:
def make_retention_dataset(crop_size=256, retain_n=200, forget_repeat=8, seed=SEED):
    with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
        coco = json.load(f)

    img_by_id = {im["id"]: im for im in coco["images"]}
    records = []

    for _ in range(forget_repeat):
        for ann in coco["annotations"]:
            im = img_by_id[ann["image_id"]]
            x, y, w, h = ann["bbox"]
            cx, cy = x + w / 2, y + h / 2
            x0 = int(np.clip(cx - crop_size / 2, 0, IMG_W - crop_size))
            y0 = int(np.clip(cy - crop_size / 2, 0, IMG_H - crop_size))

            records.append({
                "file_name": str(Path(UNLEARN_DIR) / im["file_name"]),
                "image_id": f"forget_{ann['id']}_{_}",
                "height": crop_size,
                "width": crop_size,
                "crop_xy": (x0, y0),
                "annotations": [],
            })

    teacher = build_predictor(POISONED_WEIGHTS, score_thresh=0.25)

    for i, img_path in enumerate(tqdm(sample_files(test_files, retain_n, seed), desc="Pseudo-labels")):
        records.append({
            "file_name": str(img_path),
            "image_id": f"retain_{i}",
            "height": IMG_H,
            "width": IMG_W,
            "annotations": get_teacher_annotations(teacher, img_path),
        })

    return records

def register_retention_dataset(crop_size=256, retain_n=200, forget_repeat=8):
    if RETENTION_DATASET in DatasetCatalog:
        DatasetCatalog.remove(RETENTION_DATASET)

    records = make_retention_dataset(crop_size, retain_n, forget_repeat)
    DatasetCatalog.register(RETENTION_DATASET, lambda: records)
    MetadataCatalog.get(RETENTION_DATASET).set(thing_classes=["object"])

    n_forget = sum("crop_xy" in r for r in records)
    n_retain = len(records) - n_forget
    print("Registered:", len(records), "forget:", n_forget, "retain:", n_retain)

In [ ]:
class RetentionMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])

        if "crop_xy" in dataset_dict:
            x0, y0 = dataset_dict["crop_xy"]
            image = image[y0:y0 + dataset_dict["height"], x0:x0 + dataset_dict["width"]]

        annos = dataset_dict.get("annotations", [])
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances(annos, image.shape[:2])
        return dataset_dict

class RetentionTrainer(UnlearnTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = RetentionMapper(cfg, is_train=True, augmentations=[])
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

In [ ]:
def build_retention_cfg(name, lr, iters):
    cfg = build_base_cfg(POISONED_WEIGHTS, output_dir=f"/kaggle/working/{name}")
    cfg.DATASETS.TRAIN = (RETENTION_DATASET,)
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = lr
    cfg.SOLVER.MAX_ITER = iters
    cfg.SOLVER.STEPS = []
    return cfg

def train_retention_model(name, lr, iters, crop_size=256, retain_n=200, forget_repeat=8):
    register_retention_dataset(crop_size, retain_n, forget_repeat)
    cfg = build_retention_cfg(name, lr, iters)
    trainer = RetentionTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return str(Path(cfg.OUTPUT_DIR) / "model_final.pth")

def run_retention_experiments(experiments):
    rows = []
    for exp in experiments:
        weights = train_retention_model(**exp)
        rows.append({"name": exp["name"], "lr": exp["lr"], "iters": exp["iters"], **evaluate_model(weights)})
    return pd.DataFrame(rows)

In [ ]:
retention_experiments = [
    # {"name": "retain_crop256_lr1.5e-4_iter20", "lr": 1.5e-4, "iters": 20, "crop_size": 256, "retain_n": 200, "forget_repeat": 8},
    {"name": "retain_crop256_lr2e-4_iter20", "lr": 2e-4, "iters": 20, "crop_size": 256, "retain_n": 200, "forget_repeat": 8},
]
retention_results_df = run_retention_experiments(retention_experiments)
retention_results_df

Retention-aware unlearning preserved normal detections, but the retain signal dominated the forget signal.

# 3. Submission

In [ ]:
def make_submission(predictor, files=test_files, path=SUBMISSION_PATH):
    rows = []

    for img_path in tqdm(files, desc="Inference"):
        out = predictor(load_image(img_path))["instances"].to("cpu")
        boxes, scores = out.pred_boxes.tensor.numpy(), out.scores.numpy()
        parts = []
        for (x1, y1, x2, y2), s in zip(boxes, scores):
            x1, y1 = float(np.clip(x1, 0, IMG_W)), float(np.clip(y1, 0, IMG_H))
            x2, y2 = float(np.clip(x2, 0, IMG_W)), float(np.clip(y2, 0, IMG_H))
            w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
            if w > 0 and h > 0:
                parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])
        rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

    submission = pd.DataFrame(rows)
    submission.insert(0, "id", range(len(submission)))
    submission.to_csv(path, index=False)
    print("Saved:", path, submission.shape)
    return submission

In [ ]:
# classifier_only_experiments = [
#     {"name": "cls_only_lr5e-4_iter20", "lr": 5e-4, "iters": 20},
# ]

# _ = run_classifier_only_experiments(classifier_only_experiments)

In [ ]:
BEST_MODEL_PATH = "/kaggle/working/lr2.5e-4_iter20/model_final.pth"
BEST_THRESHOLD = 0.22

predictor = build_predictor(BEST_MODEL_PATH, score_thresh=BEST_THRESHOLD)
submission = make_submission(predictor)
submission.head()